# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [13]:


### Finding 1: Growing pages were longer and younger than declining pages

# The paper reports that the rising-impression cohort averaged 3.2K words and 184 days old,
# versus 2.3K words and 230 days for the declining cohort.
# The methodology question I would ask is: **how is the growth/decline label defined,
# and does the comparison establish direction rather than cause?**
# The paper discloses that trend direction comes from a 30-day versus previous-30-day impression comparison and describes the result as observational.
# That supports a descriptive cohort comparison, but it does not show that adding words or reducing age caused growth; topic, client, and selection differences could contribute.

### Finding 2: Recently refreshed older pages had higher health and impressions

# The paper reports higher health and impressions for older pages refreshed within 30 days.
# My methodology question is: **were refreshed and unrefreshed older pages comparable before the update,
# and does the health-score construction overlap the evidence used for the claim?
# ** A simple cross-sectional comparison can be useful for prioritizing a review,
# but pages selected for refresh may already differ in visibility, editorial investment, or demand.
# A matched pre/post design with an untouched comparison group would support a stronger directional claim.
# The paper itself appropriately notes that its evidence is observational.


## 2. My model under an honest split (before/after)

The “before” result uses a random page split, which can mix pages from the same client across train and test. The “after” result holds out entire clients, which better tests whether the clustering structure transfers to a previously unseen site. Metrics are calculated on a deterministic sample of each split to keep this notebook fast and comparable.


In [14]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_samples, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_STATE = 42
METRIC_SAMPLE_SIZE = 5_000

def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data/raw/content_refresh_anonymized.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

# Works both in the local repository and in Colab after uploading the starter CSV to /content.
COLAB_DATA_PATH = Path("/content/content_refresh_anonymized.csv")
if COLAB_DATA_PATH.exists():
    ROOT = Path("/content")
    DATA_PATH = COLAB_DATA_PATH
else:
    ROOT = find_repo_root()
    DATA_PATH = ROOT / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

FEATURE_COLS = [
    "log_impressions_90d", "avg_position_clean", "days_since_last_update",
    "content_age_days", "engagement_rate", "scroll_rate_filled",
    "has_scroll_rate", "word_count_filled", "has_word_count",
]

base = df.copy()
base["log_impressions_90d"] = np.log1p(base["impressions_90d"])
base["avg_position_clean"] = base["avg_position"].replace(0, np.nan)
base["word_count_filled"] = base["word_count"].fillna(0)
base["has_word_count"] = base["word_count"].notna().astype(int)
base["has_scroll_rate"] = base["scroll_rate"].notna().astype(int)

print(f"Loaded {len(base):,} pages from {base['client_id'].nunique()} pseudonymized clients.")
print("Lane: descriptive K-Means clustering; no outcome label is predicted.")


Loaded 30,000 pages from 32 pseudonymized clients.
Lane: descriptive K-Means clustering; no outcome label is predicted.


In [15]:
def metric_rows(X, labels, max_rows=METRIC_SAMPLE_SIZE):
    # Use one deterministic row sample for all three metrics on a split.
    if len(X) <= max_rows:
        return X, labels
    rng = np.random.default_rng(RANDOM_STATE)
    take = rng.choice(len(X), size=max_rows, replace=False)
    return X[take], labels[take]

def cluster_metrics(X, labels):
    X_eval, labels_eval = metric_rows(np.asarray(X), np.asarray(labels))
    n_clusters = len(np.unique(labels_eval))
    if n_clusters < 2 or n_clusters >= len(labels_eval):
        return {"silhouette": np.nan, "calinski_harabasz": np.nan, "davies_bouldin": np.nan, "n_clusters": n_clusters, "metric_rows": len(labels_eval)}
    return {
        "silhouette": silhouette_score(X_eval, labels_eval),
        "calinski_harabasz": calinski_harabasz_score(X_eval, labels_eval),
        "davies_bouldin": davies_bouldin_score(X_eval, labels_eval),
        "n_clusters": n_clusters,
        "metric_rows": len(labels_eval),
    }

def run_split(split_name, train_mask, test_mask):
    # Fit all preprocessing and K-Means on train rows; assess on held-out rows.
    work = base.copy()
    position_median = work.loc[train_mask, "avg_position_clean"].median()
    scroll_median = work.loc[train_mask, "scroll_rate"].median()
    work["avg_position_clean"] = work["avg_position_clean"].fillna(position_median)
    work["scroll_rate_filled"] = work["scroll_rate"].fillna(scroll_median)

    X_train_raw = work.loc[train_mask, FEATURE_COLS].astype(float)
    X_test_raw = work.loc[test_mask, FEATURE_COLS].astype(float)
    assert X_train_raw.isna().sum().sum() == 0
    assert X_test_raw.isna().sum().sum() == 0

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    # Transparent baseline: no fitted parameters.
    work["baseline_group"] = (
        work["freshness_tier"].astype(str) + "|" +
        work["impression_tier"].astype(str) + "|" +
        work["position_tier"].astype(str)
    )
    baseline_codes = {name: number for number, name in enumerate(sorted(work["baseline_group"].unique()))}
    baseline_train = work.loc[train_mask, "baseline_group"].map(baseline_codes).to_numpy()
    baseline_test = work.loc[test_mask, "baseline_group"].map(baseline_codes).to_numpy()

    search = []
    for k in range(4, 9):
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        train_labels = model.fit_predict(X_train)
        search.append({"k": k, **cluster_metrics(X_train, train_labels)})
    k_search = pd.DataFrame(search).sort_values("silhouette", ascending=False)
    best_k = int(k_search.iloc[0]["k"])

    model = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
    train_labels = model.fit_predict(X_train)
    test_labels = model.predict(X_test)

    results = pd.DataFrame([
        {"validation": split_name, "method": "tier_cross_baseline", "split": "test", **cluster_metrics(X_test, baseline_test)},
        {"validation": split_name, "method": f"kmeans_k={best_k}", "split": "test", **cluster_metrics(X_test, test_labels)},
    ])
    test_df = work.loc[test_mask].copy()
    test_df["kmeans_cluster"] = test_labels
    test_df["silhouette"] = silhouette_samples(X_test, test_labels)
    return {"results": results, "best_k": best_k, "k_search": k_search, "test_df": test_df, "scaler": scaler, "model": model}

# BEFORE: random page split. It can share a client between train and test.
row_index = np.arange(len(base))
random_train_index, random_test_index = train_test_split(row_index, test_size=0.20, random_state=RANDOM_STATE)
random_train_mask = pd.Series(False, index=base.index)
random_train_mask.iloc[random_train_index] = True
random_test_mask = ~random_train_mask

# AFTER: client holdout. Entire clients are unseen in test.
clients = np.array(sorted(base["client_id"].unique()))
rng = np.random.default_rng(RANDOM_STATE)
rng.shuffle(clients)
cut = int(len(clients) * 0.8)
train_clients = set(clients[:cut])
group_train_mask = base["client_id"].isin(train_clients)
group_test_mask = ~group_train_mask

random_run = run_split("random_page_split_before", random_train_mask, random_test_mask)
group_run = run_split("grouped_client_split_after", group_train_mask, group_test_mask)

comparison = pd.concat([random_run["results"], group_run["results"]], ignore_index=True)
print("Before/after validation comparison (test rows only):")
display(comparison.round(3))
print(f"Random split: {random_train_mask.sum():,} train rows / {random_test_mask.sum():,} test rows; clients overlap by design.")
print(f"Grouped split: {group_train_mask.sum():,} train rows / {group_test_mask.sum():,} test rows; {len(train_clients)} train clients / {len(clients) - len(train_clients)} held-out clients.")
print(f"K selection: random split chose k={random_run['best_k']}; grouped split chose k={group_run['best_k']}.")


Before/after validation comparison (test rows only):


,validation,method,split,silhouette,calinski_harabasz,davies_bouldin,n_clusters,metric_rows
0,random_page_split_before,tier_cross_baseline,test,-0.237,59.085,3.356,55,5000
1,random_page_split_before,kmeans_k=6,test,0.293,1162.820,1.098,6,5000
2,grouped_client_split_after,tier_cross_baseline,test,-0.249,27.989,4.306,41,5000
3,grouped_client_split_after,kmeans_k=8,test,0.422,5361.371,1.232,8,5000


Random split: 24,000 train rows / 6,000 test rows; clients overlap by design.
Grouped split: 19,541 train rows / 10,459 test rows; 25 train clients / 7 held-out clients.
K selection: random split chose k=6; grouped split chose k=8.


## 3. Leakage audit

This is descriptive clustering, so there is no prediction label. I still audit for the same dangers: trend-derived fields, existing product decisions, identifiers, and accidental use of outcome-like fields. The 90-day metrics describe the current snapshot; they are not used to claim future performance.

In [16]:
forbidden = {
    "trend_direction", "trend_pct", "is_declining_label",  # label-derived trend fields
    "health_score", "priority_score", "action_type",        # product/system decisions or composites
    "content_id", "client_id",                               # identifiers, never model features
}

audit = pd.DataFrame([
    {"check": "Feature list excludes trend-derived fields", "passed": not bool(set(FEATURE_COLS) & {"trend_direction", "trend_pct", "is_declining_label"})},
    {"check": "Feature list excludes product flags and scores", "passed": not bool(set(FEATURE_COLS) & {"health_score", "priority_score", "action_type"})},
    {"check": "Feature list excludes IDs", "passed": not bool(set(FEATURE_COLS) & {"content_id", "client_id"})},
    {"check": "Position zero treated as missing, not rank zero", "passed": "avg_position_clean" in FEATURE_COLS},
    {"check": "Systematic scroll/word-count missingness retained with flags", "passed": {"has_scroll_rate", "has_word_count"}.issubset(FEATURE_COLS)},
    {"check": "Preprocessing fit on train rows only", "passed": True},
])
assert audit["passed"].all()
display(audit)
print("Forbidden fields present in data but excluded from features:", sorted(forbidden & set(base.columns)))
print("Result: no label-derived, product-derived, or identifier fields are used by K-Means.")


,check,passed
0,Feature list excludes trend-derived fields,True
1,Feature list excludes product flags and scores,True
2,Feature list excludes IDs,True
3,"Position zero treated as missing, not rank zero",True
4,Systematic scroll/word-count missingness retai...,True
5,Preprocessing fit on train rows only,True


Forbidden fields present in data but excluded from features: ['client_id', 'content_id', 'trend_direction', 'trend_pct']
Result: no label-derived, product-derived, or identifier fields are used by K-Means.


## 4. Claim rewrite

For clustering, a low silhouette value is an ambiguity case, not a wrong prediction: the page lies near two cluster boundaries. The examples below use only pseudonymous case numbers and numeric snapshot features.


In [17]:
test_df = group_run["test_df"].copy()
hard_cases = test_df.nsmallest(3, "silhouette").reset_index(drop=True)
failure_examples = hard_cases[[
    "kmeans_cluster", "silhouette", "impressions_90d", "avg_position_clean",
    "days_since_last_update", "engagement_rate", "scroll_rate_filled", "word_count_filled",
]].copy()
failure_examples.insert(0, "case", ["case_1", "case_2", "case_3"])
print("Three lowest-silhouette pages on held-out clients (ambiguous cluster membership):")
display(failure_examples.round(3))

cluster_quality = (
    test_df.groupby("kmeans_cluster")["silhouette"]
    .agg(mean_silhouette="mean", minimum_silhouette="min", pages="size")
    .round(3)
    .sort_values("mean_silhouette")
)
print("Held-out cluster-boundary audit:")
display(cluster_quality)

group_test = comparison.query("validation == 'grouped_client_split_after' and method.str.startswith('kmeans')", engine="python").iloc[0]
old_claim = "K-Means finds the true content archetypes and outperforms the tier rule."
new_claim = (
    f"On this 30,000-page snapshot, K-Means with k={int(group_test['n_clusters'])} produced descriptive groupings with "
    f"a held-out-client silhouette of {group_test['silhouette']:.3f}. This is a measured, directional indication of "
    "separation in these features, not proof of universal archetypes, causal effects, or a production recommendation."
)
print("Claim to avoid:", old_claim)
print("\nSafer rewrite:", new_claim)

output_dir = ROOT / "work/outputs"
output_dir.mkdir(parents=True, exist_ok=True)
receipt = {
    "lane": "structured_content_archetype_clustering",
    "random_split_best_k": random_run["best_k"],
    "grouped_split_best_k": group_run["best_k"],
    "test_metrics": comparison.round(6).to_dict(orient="records"),
    "feature_count": len(FEATURE_COLS),
    "leakage_audit_passed": bool(audit["passed"].all()),
}
(output_dir / "validation_audit_metrics.json").write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
print("Wrote work/outputs/validation_audit_metrics.json")


Three lowest-silhouette pages on held-out clients (ambiguous cluster membership):


,case,kmeans_cluster,silhouette,impressions_90d,avg_position_clean,days_since_last_update,engagement_rate,scroll_rate_filled,word_count_filled
0,case_1,1,-0.491,46437,6.7,89,3.51,8.47,2490.0
1,case_2,6,-0.442,25268,5.2,104,2.50,4.55,2956.0
2,case_3,6,-0.440,40342,3.8,104,0.46,2.73,2849.0


Held-out cluster-boundary audit:


,mean_silhouette,minimum_silhouette,pages
kmeans_cluster,,,
6,-0.054,-0.442,296
0,0.149,-0.306,61
7,0.296,-0.213,646
2,0.310,-0.262,351
1,0.326,-0.491,2502
5,0.377,-0.086,117
3,0.497,-0.086,6374
4,0.971,0.833,112


Claim to avoid: K-Means finds the true content archetypes and outperforms the tier rule.

Safer rewrite: On this 30,000-page snapshot, K-Means with k=8 produced descriptive groupings with a held-out-client silhouette of 0.422. This is a measured, directional indication of separation in these features, not proof of universal archetypes, causal effects, or a production recommendation.
Wrote work/outputs/validation_audit_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.